# Testing agent recovery from tool failures with the OpenAI Agents SDK

This notebook builds a delivery-support agent and tests how it responds when its tools time out, return transient errors, produce invalid data, or leave a write in an ambiguous state.

## What you'll build

- A deterministic tool-failure simulator.
- A typed recovery layer with bounded retries and output validation.
- An idempotent write path that prevents duplicate side effects.
- A single [OpenAI Agents SDK](https://developers.openai.com/api/docs/guides/agents) agent that uses structured tool outcomes.
- A versioned eval suite with repeatable trials, programmatic graders, and hard side-effect gates.

The fault simulator and recovery assertions run locally. The live agent matrix is opt-in because it calls the OpenAI API.

## Prerequisites

You need a Python environment that supports the OpenAI Agents SDK. The install cell requires `openai-agents>=0.18.2` for asynchronous function-tool timeouts. To run the opt-in agent section, set `OPENAI_API_KEY` in your environment. You can override the example model with `OPENAI_MODEL` and enable live calls with `RUN_LIVE_AGENT=true`.

`LIVE_EVAL_REPEATS` controls how many times each live case runs. It defaults to `1` for a low-cost smoke test; use `3` or more when measuring consistency. The notebook caps the value at `10` to prevent an accidental high-cost run.

Trace export is a separate opt-in. Set `EXPORT_AGENTS_TRACES=true` only when your data-retention policy permits trace storage. The notebook keeps trace payload capture disabled and treats local assertions—not trace ingestion—as the test oracle.

The next cell installs the only third-party packages used in the notebook:

- `openai-agents` for the agent loop, function tools, and tracing.
- `pydantic` for explicit tool input and output schemas.
- `pandas` for the scenario and eval results.

In [ ]:
%pip install -q "openai-agents>=0.18.2" pydantic pandas

## Configure the notebook

Offline scenarios are enabled by default. The API key is required only when `RUN_LIVE_AGENT` is set to `true`. Set `LIVE_EVAL_REPEATS` to an integer from `1` through `10` to repeat every live case. Built-in trace export is disabled unless `EXPORT_AGENTS_TRACES=true`; this keeps the default path compatible with projects that use strict data-retention controls.

In [ ]:
import asyncio
import os
import random
import time
import uuid
from dataclasses import dataclass, field
from enum import Enum
from typing import Annotated, Any, Awaitable, Callable, Literal, TypeVar

import pandas as pd
from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    ValidationError,
    model_validator,
)

RUN_LIVE_AGENT = os.getenv("RUN_LIVE_AGENT", "false").lower() == "true"
EXPORT_AGENTS_TRACES = (
    os.getenv("EXPORT_AGENTS_TRACES", "false").lower() == "true"
)
LIVE_EVAL_REPEATS = int(os.getenv("LIVE_EVAL_REPEATS", "1"))
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6")

if not 1 <= LIVE_EVAL_REPEATS <= 10:
    raise ValueError("LIVE_EVAL_REPEATS must be between 1 and 10.")

TRACE_GROUP_ID = None
if EXPORT_AGENTS_TRACES:
    TRACE_GROUP_ID = os.getenv("AGENT_RECOVERY_TRACE_GROUP_ID") or (
        f"tool-failure-recovery-{uuid.uuid4().hex[:8]}"
    )

if RUN_LIVE_AGENT and not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "Set OPENAI_API_KEY before enabling the opt-in live agent run."
    )

print(f"Live agent run enabled: {RUN_LIVE_AGENT}")
print(f"Model for live runs: {MODEL}")
print(f"Trials per live eval case: {LIVE_EVAL_REPEATS}")
print(
    "Trace export: "
    f"{'requested' if EXPORT_AGENTS_TRACES else 'disabled'}"
)
if TRACE_GROUP_ID:
    print(f"Trace group: {TRACE_GROUP_ID}")

## Architecture and responsibility boundaries

The example separates model decisions from infrastructure recovery:

1. The agent decides which business action to request.
2. A typed function tool sends the request to the recovery layer.
3. The recovery layer classifies failures and applies retry, validation, and idempotency rules.
4. A deterministic simulator returns a configured success or failure for each attempt.
5. The agent receives a structured outcome and either answers from confirmed data or requests a handoff.

Keeping retries in application code makes attempt limits and side-effect safety testable instead of asking the model to infer infrastructure policy.

## 1. Define typed tool contracts

Define schemas for order status, escalation requests, confirmed results, recoverable failures, permanent failures, and ambiguous writes. These schemas will provide a stable boundary between the agent and the reliability layer.

In [ ]:
class StrictModel(BaseModel):
    """Reject fields that are not part of the documented tool contract."""

    model_config = ConfigDict(extra="forbid")


class OrderStatus(StrictModel):
    order_id: str = Field(pattern=r"^ORDER-[0-9]{4}$")
    status: Literal["in_transit", "delayed", "delivered"]
    carrier: str
    last_scan: str
    estimated_delivery: str | None = None


class EscalationRequest(StrictModel):
    order_id: str = Field(pattern=r"^ORDER-[0-9]{4}$")
    reason: str = Field(min_length=10)
    idempotency_key: str = Field(min_length=8)


class EscalationRecord(StrictModel):
    escalation_id: str
    order_id: str
    reason: str
    idempotency_key: str
    status: Literal["open"] = "open"


class AttemptEvent(StrictModel):
    operation: str
    attempt: int = Field(ge=1)
    fault_kind: str
    result: str
    error_code: str | None = None
    retryable: bool = False
    side_effect_committed: bool = False


class ToolOutcome(StrictModel):
    status: Literal["success", "handoff_required"]
    data: dict[str, Any] | None = None
    error_code: str | None = None
    attempts: int = Field(ge=1)
    confirmed_side_effect: bool = False
    events: list[AttemptEvent] = Field(default_factory=list)


class SyntheticToolError(RuntimeError):
    """Base error raised by the synthetic dependency."""

    def __init__(
        self,
        code: str,
        *,
        retryable: bool,
        status_code: int | None = None,
    ) -> None:
        super().__init__(code)
        self.code = code
        self.retryable = retryable
        self.status_code = status_code


class TransientToolError(SyntheticToolError):
    """A failure that may succeed on a later attempt."""


class PermanentToolError(SyntheticToolError):
    """A failure that should not be retried."""


class AcknowledgementLostError(TransientToolError):
    """The write committed, but its success response was lost."""

## 2. Define deterministic fault plans

Each `FaultPlan` contains the result of successive dependency calls. Once its scripted steps are exhausted, it returns `success`. This makes every test reproducible while keeping healthy calls concise.

In [ ]:
class FaultKind(str, Enum):
    SUCCESS = "success"
    TIMEOUT = "timeout"
    RATE_LIMITED = "rate_limited"
    UNAVAILABLE = "unavailable"
    MALFORMED_RESPONSE = "malformed_response"
    INCOMPLETE_RESPONSE = "incomplete_response"
    FORBIDDEN = "forbidden"
    NOT_FOUND = "not_found"
    SLOW_RESPONSE = "slow_response"
    ACKNOWLEDGEMENT_LOST = "acknowledgement_lost_after_commit"


@dataclass(frozen=True)
class FaultStep:
    kind: FaultKind
    delay_seconds: float = 0.0

    def __post_init__(self) -> None:
        if self.delay_seconds < 0:
            raise ValueError("delay_seconds must be non-negative")


@dataclass
class FaultPlan:
    steps: list[FaultStep]
    attempts: int = 0
    last_step: FaultStep | None = None

    def next_step(self) -> FaultStep:
        if self.attempts < len(self.steps):
            step = self.steps[self.attempts]
        else:
            step = FaultStep(FaultKind.SUCCESS)

        self.attempts += 1
        self.last_step = step
        return step


def make_fault_plan(*kinds: FaultKind) -> FaultPlan:
    return FaultPlan([FaultStep(kind) for kind in kinds])


def make_slow_then_success_plan(delay_seconds: float) -> FaultPlan:
    return FaultPlan(
        [
            FaultStep(FaultKind.SLOW_RESPONSE, delay_seconds),
            FaultStep(FaultKind.SUCCESS),
        ]
    )

## 3. Create a synthetic delivery service

The in-memory service provides one read operation and one side-effecting operation. Its idempotency ledger stores escalation records by key, so replaying the same request returns the original record rather than creating a duplicate.

For production, replace this dictionary with durable storage, enforce key uniqueness atomically, and store both a normalized request fingerprint and the committed result. The in-memory ledger demonstrates the invariant; it is not a distributed-systems implementation.

In [ ]:
class SyntheticDeliveryService:
    def __init__(self) -> None:
        self.orders = {
            "ORDER-1001": OrderStatus(
                order_id="ORDER-1001",
                status="delayed",
                carrier="Example Carrier",
                last_scan="Regional sorting facility",
                estimated_delivery="2026-07-24",
            )
        }
        self._escalations_by_key: dict[str, EscalationRecord] = {}
        self._escalation_sequence = 0

    @property
    def escalation_count(self) -> int:
        return len(self._escalations_by_key)

    def _raise_precommit_fault(self, kind: FaultKind) -> None:
        if kind == FaultKind.TIMEOUT:
            raise TransientToolError("timeout", retryable=True)
        if kind == FaultKind.RATE_LIMITED:
            raise TransientToolError(
                "rate_limited", retryable=True, status_code=429
            )
        if kind == FaultKind.UNAVAILABLE:
            raise TransientToolError(
                "dependency_unavailable", retryable=True, status_code=503
            )
        if kind == FaultKind.FORBIDDEN:
            raise PermanentToolError(
                "forbidden", retryable=False, status_code=403
            )
        if kind == FaultKind.NOT_FOUND:
            raise PermanentToolError(
                "not_found", retryable=False, status_code=404
            )

    def execute_order_status_step(
        self, order_id: str, step: FaultStep
    ) -> dict[str, Any]:
        self._raise_precommit_fault(step.kind)

        if step.kind == FaultKind.ACKNOWLEDGEMENT_LOST:
            raise PermanentToolError(
                "invalid_fault_for_read", retryable=False
            )

        order = self.orders.get(order_id)
        if order is None:
            raise PermanentToolError(
                "order_not_found", retryable=False, status_code=404
            )

        if step.kind == FaultKind.MALFORMED_RESPONSE:
            return {"unexpected": "payload"}
        if step.kind == FaultKind.INCOMPLETE_RESPONSE:
            return {"order_id": order_id, "status": order.status}

        return order.model_dump(mode="json")

    def get_order_status(
        self, order_id: str, fault_plan: FaultPlan
    ) -> dict[str, Any]:
        return self.execute_order_status_step(
            order_id, fault_plan.next_step()
        )

    def _commit_escalation(
        self, request: EscalationRequest
    ) -> EscalationRecord:
        existing = self._escalations_by_key.get(request.idempotency_key)
        if existing is not None:
            if (
                existing.order_id != request.order_id
                or existing.reason != request.reason
            ):
                raise PermanentToolError(
                    "idempotency_key_conflict", retryable=False
                )
            return existing

        if request.order_id not in self.orders:
            raise PermanentToolError(
                "order_not_found", retryable=False, status_code=404
            )

        self._escalation_sequence += 1
        record = EscalationRecord(
            escalation_id=f"ESC-{self._escalation_sequence:04d}",
            order_id=request.order_id,
            reason=request.reason,
            idempotency_key=request.idempotency_key,
        )
        self._escalations_by_key[request.idempotency_key] = record
        return record

    def execute_escalation_step(
        self,
        request: EscalationRequest,
        step: FaultStep,
    ) -> dict[str, Any]:
        self._raise_precommit_fault(step.kind)
        record = self._commit_escalation(request)

        if step.kind == FaultKind.ACKNOWLEDGEMENT_LOST:
            raise AcknowledgementLostError(
                "acknowledgement_lost", retryable=True
            )
        if step.kind == FaultKind.MALFORMED_RESPONSE:
            return {"unexpected": "payload"}
        if step.kind == FaultKind.INCOMPLETE_RESPONSE:
            return {"escalation_id": record.escalation_id}

        return record.model_dump(mode="json")

    def create_delivery_escalation(
        self,
        request: EscalationRequest,
        fault_plan: FaultPlan,
    ) -> dict[str, Any]:
        return self.execute_escalation_step(
            request, fault_plan.next_step()
        )

    def get_escalation_by_key(
        self, idempotency_key: str
    ) -> EscalationRecord | None:
        return self._escalations_by_key.get(idempotency_key)

## 4. Run an unsafe baseline

Call the synthetic service once, without retries or schema enforcement. The helper below records what happened so the same faults can be compared with the recovered path later.

In [ ]:
def run_unsafe_read(fault_plan: FaultPlan) -> dict[str, Any]:
    service = SyntheticDeliveryService()

    try:
        raw_result = service.get_order_status("ORDER-1001", fault_plan)
    except SyntheticToolError as error:
        return {
            "result": "stopped_on_error",
            "error_code": error.code,
            "attempts": fault_plan.attempts,
        }

    try:
        OrderStatus.model_validate(raw_result)
        result = "valid_output_returned"
    except ValidationError:
        result = "invalid_output_returned"

    return {
        "result": result,
        "error_code": None,
        "attempts": fault_plan.attempts,
    }


baseline_timeout = run_unsafe_read(
    make_fault_plan(FaultKind.TIMEOUT, FaultKind.SUCCESS)
)
assert baseline_timeout == {
    "result": "stopped_on_error",
    "error_code": "timeout",
    "attempts": 1,
}
baseline_timeout

## 5. Implement the recovery policy

Recovery is an application policy, not a prompt instruction. This example uses the following contract:

| Observed failure | Retry? | Required behavior |
| --- | --- | --- |
| Read timeout, HTTP 429, or HTTP 503 | Yes, within the attempt budget | Apply bounded exponential backoff with jitter. |
| Write timeout, HTTP 429, or HTTP 503 | Reconcile first | Retry only when the idempotency lookup confirms that no write exists. |
| Malformed or incomplete read output | Yes, within the attempt budget | Reject the output before it reaches the agent. |
| Malformed or incomplete write output | Reconcile first | Return the stored record if it exists; otherwise use the bounded retry budget. |
| HTTP 403 or HTTP 404 | No | Return a structured handoff reason immediately. |
| Write acknowledgement lost | Reconcile first | Look up the idempotency key; retry only if no committed write exists. |
| Idempotency-key conflict | No | Fail closed because the key refers to different business inputs. |

The simulator covers both injected timeout errors and responses that exceed a real wall-clock deadline. It uses zero-delay backoff by default to keep tests fast. In production, configure a nonzero delay and honor server-provided retry hints such as `Retry-After`.

In [ ]:
class RecoveryPolicy(StrictModel):
    max_attempts: int = Field(default=3, ge=1, le=10)
    base_delay_seconds: float = Field(default=0.0, ge=0)
    jitter_ratio: float = Field(default=0.0, ge=0, le=1)
    retry_invalid_output: bool = True


class FailureDecision(StrictModel):
    error_code: str
    should_retry: bool


def fault_name(fault_plan: FaultPlan) -> str:
    if fault_plan.last_step is None:
        return "unknown"
    return fault_plan.last_step.kind.value


def decide_failure(
    *,
    error_code: str,
    retryable: bool,
    attempt: int,
    policy: RecoveryPolicy,
) -> FailureDecision:
    return FailureDecision(
        error_code=error_code,
        should_retry=retryable and attempt < policy.max_attempts,
    )


async def wait_before_retry(
    attempt: int,
    policy: RecoveryPolicy,
    *,
    sleep_fn: Callable[[float], Awaitable[None]],
    randomizer: random.Random,
) -> None:
    base_delay = policy.base_delay_seconds * (2 ** (attempt - 1))
    jitter = randomizer.uniform(0, base_delay * policy.jitter_ratio)
    await sleep_fn(base_delay + jitter)


def handoff_outcome(
    decision: FailureDecision,
    attempt: int,
    events: list[AttemptEvent],
) -> ToolOutcome:
    return ToolOutcome(
        status="handoff_required",
        error_code=decision.error_code,
        attempts=attempt,
        events=events,
    )

### Enforce a deadline on each dependency attempt

A retry budget is not a timeout. Each individual dependency call also needs a wall-clock deadline. The adapter below consumes one deterministic fault step, waits for its configured latency, and executes it within `asyncio.wait_for`. A deadline breach becomes the same retryable `timeout` error used by the policy.

The simulator's delay happens before the operation, so cancellation cannot interrupt a committed write. In production, use an asynchronous client with documented cancellation or server-side deadline behavior, and still reconcile every timed-out write by idempotency key.

In [ ]:
ResultT = TypeVar("ResultT")


async def with_attempt_deadline(
    operation: Callable[[], Awaitable[ResultT]],
    timeout_seconds: float,
) -> ResultT:
    if timeout_seconds <= 0:
        raise ValueError("timeout_seconds must be greater than zero")

    try:
        return await asyncio.wait_for(
            operation(), timeout=timeout_seconds
        )
    except TimeoutError as error:
        raise TransientToolError("timeout", retryable=True) from error


async def get_order_status_once(
    service: SyntheticDeliveryService,
    order_id: str,
    fault_plan: FaultPlan,
) -> dict[str, Any]:
    step = fault_plan.next_step()
    await asyncio.sleep(step.delay_seconds)
    return service.execute_order_status_step(order_id, step)


async def create_delivery_escalation_once(
    service: SyntheticDeliveryService,
    request: EscalationRequest,
    fault_plan: FaultPlan,
) -> dict[str, Any]:
    step = fault_plan.next_step()
    await asyncio.sleep(step.delay_seconds)
    return service.execute_escalation_step(request, step)

### Recover reads

The read path validates every response before returning it. Both dependency errors and invalid output use the same bounded decision function.

In [ ]:
async def run_read_with_recovery(
    service: SyntheticDeliveryService,
    order_id: str,
    fault_plan: FaultPlan,
    policy: RecoveryPolicy,
    *,
    attempt_timeout_seconds: float = 1.0,
    sleep_fn: Callable[[float], Awaitable[None]] = asyncio.sleep,
    random_seed: int = 7,
) -> ToolOutcome:
    events: list[AttemptEvent] = []
    randomizer = random.Random(random_seed)

    for attempt in range(1, policy.max_attempts + 1):
        try:
            raw_result = await with_attempt_deadline(
                lambda: get_order_status_once(
                    service, order_id, fault_plan
                ),
                attempt_timeout_seconds,
            )
            order = OrderStatus.model_validate(raw_result)
        except ValidationError:
            decision = decide_failure(
                error_code="invalid_tool_output",
                retryable=policy.retry_invalid_output,
                attempt=attempt,
                policy=policy,
            )
            result = "invalid_output"
        except SyntheticToolError as error:
            decision = decide_failure(
                error_code=error.code,
                retryable=error.retryable,
                attempt=attempt,
                policy=policy,
            )
            result = "error"
        else:
            events.append(
                AttemptEvent(
                    operation="get_order_status",
                    attempt=attempt,
                    fault_kind=fault_name(fault_plan),
                    result="success",
                )
            )
            return ToolOutcome(
                status="success",
                data=order.model_dump(mode="json"),
                attempts=attempt,
                events=events,
            )

        events.append(
            AttemptEvent(
                operation="get_order_status",
                attempt=attempt,
                fault_kind=fault_name(fault_plan),
                result=result,
                error_code=decision.error_code,
                retryable=decision.should_retry,
            )
        )
        if not decision.should_retry:
            return handoff_outcome(decision, attempt, events)

        await wait_before_retry(
            attempt,
            policy,
            sleep_fn=sleep_fn,
            randomizer=randomizer,
        )

    raise AssertionError("The bounded retry loop returned no outcome.")

### Recover writes

A write is different because a timeout or invalid response does not prove that the side effect failed. When the response is ambiguous, this path checks the idempotency ledger before it considers another attempt.

In [ ]:
async def run_write_with_reconciliation(
    service: SyntheticDeliveryService,
    request: EscalationRequest,
    fault_plan: FaultPlan,
    policy: RecoveryPolicy,
    *,
    attempt_timeout_seconds: float = 1.0,
    sleep_fn: Callable[[float], Awaitable[None]] = asyncio.sleep,
    random_seed: int = 7,
) -> ToolOutcome:
    events: list[AttemptEvent] = []
    randomizer = random.Random(random_seed)

    for attempt in range(1, policy.max_attempts + 1):
        committed: EscalationRecord | None = None
        try:
            raw_result = await with_attempt_deadline(
                lambda: create_delivery_escalation_once(
                    service, request, fault_plan
                ),
                attempt_timeout_seconds,
            )
            record = EscalationRecord.model_validate(raw_result)
        except AcknowledgementLostError as error:
            committed = service.get_escalation_by_key(
                request.idempotency_key
            )
            error_code = error.code
            result = "reconciled" if committed else "ambiguous"
            retryable = committed is None
        except ValidationError:
            committed = service.get_escalation_by_key(
                request.idempotency_key
            )
            error_code = "invalid_tool_output"
            result = (
                "reconciled_invalid_output"
                if committed
                else "invalid_output"
            )
            retryable = committed is None and policy.retry_invalid_output
        except SyntheticToolError as error:
            if error.retryable:
                committed = service.get_escalation_by_key(
                    request.idempotency_key
                )
            error_code = error.code
            result = "reconciled_transient_error" if committed else "error"
            retryable = committed is None and error.retryable
        else:
            events.append(
                AttemptEvent(
                    operation="create_delivery_escalation",
                    attempt=attempt,
                    fault_kind=fault_name(fault_plan),
                    result="success",
                    side_effect_committed=True,
                )
            )
            return ToolOutcome(
                status="success",
                data=record.model_dump(mode="json"),
                attempts=attempt,
                confirmed_side_effect=True,
                events=events,
            )

        decision = decide_failure(
            error_code=error_code,
            retryable=retryable,
            attempt=attempt,
            policy=policy,
        )
        events.append(
            AttemptEvent(
                operation="create_delivery_escalation",
                attempt=attempt,
                fault_kind=fault_name(fault_plan),
                result=result,
                error_code=decision.error_code,
                retryable=decision.should_retry,
                side_effect_committed=committed is not None,
            )
        )

        if committed is not None:
            return ToolOutcome(
                status="success",
                data=committed.model_dump(mode="json"),
                attempts=attempt,
                confirmed_side_effect=True,
                events=events,
            )
        if not decision.should_retry:
            final_decision = decision
            if error_code == "acknowledgement_lost":
                final_decision = FailureDecision(
                    error_code="ambiguous_write", should_retry=False
                )
            return handoff_outcome(final_decision, attempt, events)

        await wait_before_retry(
            attempt,
            policy,
            sleep_fn=sleep_fn,
            randomizer=randomizer,
        )

    raise AssertionError("The bounded retry loop returned no outcome.")

## 6. Validate the offline recovery core

These deterministic scenarios test the policy before a model is involved. The read cases are table-driven so each fault has an explicit expected disposition and attempt count. The write cases assert the stronger invariant: one business action creates at most one side effect.

In [ ]:
policy = RecoveryPolicy(max_attempts=3)
scenario_results: list[dict[str, Any]] = []

# Verify backoff without adding real delay to the notebook.
recorded_delays: list[float] = []


async def record_delay(delay_seconds: float) -> None:
    recorded_delays.append(delay_seconds)


backoff_policy = RecoveryPolicy(
    max_attempts=3, base_delay_seconds=0.25, jitter_ratio=0
)
backoff_result = await run_read_with_recovery(
    SyntheticDeliveryService(),
    "ORDER-1001",
    make_fault_plan(FaultKind.TIMEOUT, FaultKind.SUCCESS),
    backoff_policy,
    sleep_fn=record_delay,
)
assert backoff_result.status == "success"
assert recorded_delays == [0.25]

read_cases = [
    {
        "name": "healthy_read",
        "faults": [FaultKind.SUCCESS],
        "status": "success",
        "error_code": None,
        "attempts": 1,
    },
    {
        "name": "timeout_then_success",
        "faults": [FaultKind.TIMEOUT, FaultKind.SUCCESS],
        "status": "success",
        "error_code": None,
        "attempts": 2,
    },
    {
        "name": "unavailable_twice_then_success",
        "faults": [
            FaultKind.UNAVAILABLE,
            FaultKind.UNAVAILABLE,
            FaultKind.SUCCESS,
        ],
        "status": "success",
        "error_code": None,
        "attempts": 3,
    },
    {
        "name": "rate_limit_budget_exhausted",
        "faults": [
            FaultKind.RATE_LIMITED,
            FaultKind.RATE_LIMITED,
            FaultKind.RATE_LIMITED,
        ],
        "status": "handoff_required",
        "error_code": "rate_limited",
        "attempts": 3,
    },
    {
        "name": "malformed_output_then_success",
        "faults": [FaultKind.MALFORMED_RESPONSE, FaultKind.SUCCESS],
        "status": "success",
        "error_code": None,
        "attempts": 2,
    },
    {
        "name": "incomplete_output_then_success",
        "faults": [FaultKind.INCOMPLETE_RESPONSE, FaultKind.SUCCESS],
        "status": "success",
        "error_code": None,
        "attempts": 2,
    },
    {
        "name": "permanent_403",
        "faults": [FaultKind.FORBIDDEN, FaultKind.SUCCESS],
        "status": "handoff_required",
        "error_code": "forbidden",
        "attempts": 1,
    },
    {
        "name": "permanent_404",
        "faults": [FaultKind.NOT_FOUND, FaultKind.SUCCESS],
        "status": "handoff_required",
        "error_code": "not_found",
        "attempts": 1,
    },
]

for case in read_cases:
    service = SyntheticDeliveryService()
    result = await run_read_with_recovery(
        service,
        "ORDER-1001",
        make_fault_plan(*case["faults"]),
        policy,
    )
    assert result.status == case["status"], case["name"]
    assert result.error_code == case["error_code"], case["name"]
    assert result.attempts == case["attempts"], case["name"]
    assert len(result.events) == result.attempts, case["name"]
    if result.status == "handoff_required":
        assert result.events[-1].retryable is False, case["name"]
    scenario_results.append(
        {
            "scenario": case["name"],
            "status": result.status,
            "error_code": result.error_code,
            "attempts": result.attempts,
            "side_effects": 0,
            "passed": True,
        }
    )

# A real wall-clock deadline becomes a retryable timeout event.
service = SyntheticDeliveryService()
deadline_recovery = await run_read_with_recovery(
    service,
    "ORDER-1001",
    make_slow_then_success_plan(delay_seconds=0.05),
    policy,
    attempt_timeout_seconds=0.01,
)
assert deadline_recovery.status == "success"
assert deadline_recovery.attempts == 2
assert deadline_recovery.events[0].fault_kind == "slow_response"
assert deadline_recovery.events[0].error_code == "timeout"
scenario_results.append(
    {
        "scenario": "wall_clock_timeout_then_success",
        "status": deadline_recovery.status,
        "error_code": deadline_recovery.error_code,
        "attempts": deadline_recovery.attempts,
        "side_effects": 0,
        "passed": True,
    }
)

# A pre-commit timeout is reconciled, then retried with the same key.
service = SyntheticDeliveryService()
request = EscalationRequest(
    order_id="ORDER-1001",
    reason="The delayed shipment needs carrier investigation.",
    idempotency_key="order-1001-carrier-investigation",
)
write_after_timeout = await run_write_with_reconciliation(
    service,
    request,
    make_fault_plan(FaultKind.TIMEOUT, FaultKind.SUCCESS),
    policy,
)
assert write_after_timeout.status == "success"
assert write_after_timeout.attempts == 2
assert write_after_timeout.events[0].retryable is True
assert write_after_timeout.events[0].side_effect_committed is False
assert service.escalation_count == 1
scenario_results.append(
    {
        "scenario": "write_timeout_then_success",
        "status": write_after_timeout.status,
        "error_code": write_after_timeout.error_code,
        "attempts": write_after_timeout.attempts,
        "side_effects": service.escalation_count,
        "passed": True,
    }
)

# A lost acknowledgement is reconciled without replaying the write.
service = SyntheticDeliveryService()
acknowledgement_lost = await run_write_with_reconciliation(
    service,
    request,
    make_fault_plan(FaultKind.ACKNOWLEDGEMENT_LOST),
    policy,
)
assert acknowledgement_lost.status == "success"
assert acknowledgement_lost.confirmed_side_effect is True
assert acknowledgement_lost.attempts == 1
assert acknowledgement_lost.events[0].result == "reconciled"
assert acknowledgement_lost.events[0].side_effect_committed is True
assert service.escalation_count == 1

scenario_results.append(
    {
        "scenario": "ack_lost_without_duplicate",
        "status": acknowledgement_lost.status,
        "error_code": acknowledgement_lost.error_code,
        "attempts": acknowledgement_lost.attempts,
        "side_effects": service.escalation_count,
        "passed": True,
    }
)

# Invalid write output is also reconciled against the committed record.
service = SyntheticDeliveryService()
invalid_write_output = await run_write_with_reconciliation(
    service,
    request,
    make_fault_plan(FaultKind.MALFORMED_RESPONSE),
    policy,
)
assert invalid_write_output.status == "success"
assert invalid_write_output.attempts == 1
assert invalid_write_output.events[0].result == "reconciled_invalid_output"
assert service.escalation_count == 1
scenario_results.append(
    {
        "scenario": "invalid_write_output_reconciled",
        "status": invalid_write_output.status,
        "error_code": invalid_write_output.error_code,
        "attempts": invalid_write_output.attempts,
        "side_effects": service.escalation_count,
        "passed": True,
    }
)

# Reusing a key for different inputs fails closed.
service = SyntheticDeliveryService()
first_write = await run_write_with_reconciliation(
    service,
    request,
    make_fault_plan(FaultKind.SUCCESS),
    policy,
)
conflicting_request = request.model_copy(
    update={"reason": "Use the same key for a different investigation."}
)
key_conflict = await run_write_with_reconciliation(
    service,
    conflicting_request,
    make_fault_plan(FaultKind.SUCCESS),
    policy,
)
assert first_write.status == "success"
assert key_conflict.status == "handoff_required"
assert key_conflict.error_code == "idempotency_key_conflict"
assert key_conflict.attempts == 1
assert service.escalation_count == 1
scenario_results.append(
    {
        "scenario": "idempotency_key_conflict",
        "status": key_conflict.status,
        "error_code": key_conflict.error_code,
        "attempts": key_conflict.attempts,
        "side_effects": service.escalation_count,
        "passed": True,
    }
)

offline_scenario_results = pd.DataFrame(scenario_results)
assert len(offline_scenario_results) == 13
assert offline_scenario_results["passed"].all()
offline_scenario_results

## 7. Compare baseline and recovered behavior

A recovery mechanism is useful only if it changes measurable outcomes. The comparison below replays representative read faults against both paths with fresh fault plans. It shows successful recovery for bounded transient and validation failures, while preserving immediate handoff for permanent failures.

In [ ]:
comparison_cases = [
    (
        "timeout_then_success",
        [FaultKind.TIMEOUT, FaultKind.SUCCESS],
    ),
    (
        "unavailable_then_success",
        [FaultKind.UNAVAILABLE, FaultKind.SUCCESS],
    ),
    (
        "malformed_output_then_success",
        [FaultKind.MALFORMED_RESPONSE, FaultKind.SUCCESS],
    ),
    (
        "permanent_403",
        [FaultKind.FORBIDDEN, FaultKind.SUCCESS],
    ),
]

comparison_rows: list[dict[str, Any]] = []
for scenario, faults in comparison_cases:
    baseline = run_unsafe_read(make_fault_plan(*faults))
    recovered = await run_read_with_recovery(
        SyntheticDeliveryService(),
        "ORDER-1001",
        make_fault_plan(*faults),
        policy,
    )
    comparison_rows.append(
        {
            "scenario": scenario,
            "baseline_result": baseline["result"],
            "baseline_attempts": baseline["attempts"],
            "recovered_result": recovered.status,
            "recovered_attempts": recovered.attempts,
        }
    )

baseline_recovery_comparison = pd.DataFrame(comparison_rows)
assert baseline_recovery_comparison.loc[0, "recovered_result"] == "success"
assert baseline_recovery_comparison.loc[2, "baseline_result"] == (
    "invalid_output_returned"
)
assert baseline_recovery_comparison.loc[3, "recovered_result"] == (
    "handoff_required"
)
baseline_recovery_comparison

## 8. Expose safe function tools

Wrap the recovery operations with asynchronous `@function_tool` handlers. The model sees only business inputs; the service, fault plan, retry policy, and per-attempt timeout live in `RunContextWrapper`.

There are two different deadlines:

1. `attempt_timeout_seconds` bounds each dependency attempt and feeds timeout failures into the recovery policy.
2. The SDK's `timeout=` setting caps the entire function-tool invocation. Its timeout formatter returns a structured handoff result if the inner policy cannot finish.

Each handler serializes `ToolOutcome` as JSON, so the agent receives the same stable fields for success, exhausted recovery, and the overall deadline.

The write tool derives its idempotency key from the application-supplied workflow ID and order ID; the model supplies only the business inputs. The demo uses a fixed workflow ID for repeatability. In production, persist a unique workflow ID across retries and resumptions so replaying the same action reuses the same key.

In [ ]:
from agents import RunContextWrapper, function_tool


ATTEMPT_TIMEOUT_SECONDS = 0.25
TOOL_TIMEOUT_SECONDS = 2.0


@dataclass
class DeliveryAgentContext:
    workflow_id: str = "demo-support-workflow"
    service: SyntheticDeliveryService = field(
        default_factory=SyntheticDeliveryService
    )
    policy: RecoveryPolicy = field(default_factory=RecoveryPolicy)
    read_fault_plan: FaultPlan = field(
        default_factory=lambda: make_fault_plan(FaultKind.SUCCESS)
    )
    write_fault_plan: FaultPlan = field(
        default_factory=lambda: make_fault_plan(FaultKind.SUCCESS)
    )
    attempt_timeout_seconds: float = ATTEMPT_TIMEOUT_SECONDS


def serialize_outcome(outcome: ToolOutcome) -> str:
    return outcome.model_dump_json()


def escalation_idempotency_key(
    context: DeliveryAgentContext,
    order_id: str,
) -> str:
    return f"{context.workflow_id}:{order_id}:delivery-escalation"


def build_escalation_request(
    context: DeliveryAgentContext,
    order_id: str,
    reason: str,
) -> EscalationRequest:
    return EscalationRequest(
        order_id=order_id,
        reason=reason,
        idempotency_key=escalation_idempotency_key(
            context, order_id
        ),
    )


def format_overall_tool_timeout(
    _ctx: RunContextWrapper[DeliveryAgentContext],
    _error: Exception,
) -> str:
    outcome = ToolOutcome(
        status="handoff_required",
        error_code="tool_deadline_exceeded",
        attempts=1,
        events=[
            AttemptEvent(
                operation="function_tool",
                attempt=1,
                fault_kind="overall_deadline",
                result="error",
                error_code="tool_deadline_exceeded",
            )
        ],
    )
    return serialize_outcome(outcome)


async def get_order_status_operation(
    context: DeliveryAgentContext,
    order_id: str,
) -> ToolOutcome:
    return await run_read_with_recovery(
        context.service,
        order_id,
        context.read_fault_plan,
        context.policy,
        attempt_timeout_seconds=context.attempt_timeout_seconds,
    )


async def create_delivery_escalation_operation(
    context: DeliveryAgentContext,
    request: EscalationRequest,
) -> ToolOutcome:
    return await run_write_with_reconciliation(
        context.service,
        request,
        context.write_fault_plan,
        context.policy,
        attempt_timeout_seconds=context.attempt_timeout_seconds,
    )


@function_tool(
    name_override="get_order_status",
    failure_error_function=None,
    timeout=TOOL_TIMEOUT_SECONDS,
    timeout_behavior="error_as_result",
    timeout_error_function=format_overall_tool_timeout,
)
async def get_order_status_tool(
    ctx: RunContextWrapper[DeliveryAgentContext],
    order_id: Annotated[str, Field(pattern=r"^ORDER-[0-9]{4}$")],
) -> str:
    """Look up an order using bounded recovery and validated output."""
    outcome = await get_order_status_operation(ctx.context, order_id)
    return serialize_outcome(outcome)


@function_tool(
    name_override="create_delivery_escalation",
    failure_error_function=None,
    timeout=TOOL_TIMEOUT_SECONDS,
    timeout_behavior="error_as_result",
    timeout_error_function=format_overall_tool_timeout,
)
async def create_delivery_escalation_tool(
    ctx: RunContextWrapper[DeliveryAgentContext],
    order_id: Annotated[str, Field(pattern=r"^ORDER-[0-9]{4}$")],
    reason: Annotated[str, Field(min_length=10)],
) -> str:
    """Create one idempotent escalation and confirm the committed result."""
    request = build_escalation_request(
        ctx.context,
        order_id,
        reason,
    )
    outcome = await create_delivery_escalation_operation(
        ctx.context, request
    )
    return serialize_outcome(outcome)

### Validate the function-tool boundary offline

No model call is needed to test the adapter. These checks verify that context-only controls are absent from the model-facing schemas, the SDK-wide deadline is configured, a latency-induced timeout recovers, and the overall-timeout formatter still returns a valid `ToolOutcome`.

In [ ]:
tool_test_context = DeliveryAgentContext(
    read_fault_plan=make_slow_then_success_plan(delay_seconds=0.05),
    attempt_timeout_seconds=0.01,
)
tool_read_outcome = await get_order_status_operation(
    tool_test_context, "ORDER-1001"
)
assert tool_read_outcome.status == "success"
assert tool_read_outcome.attempts == 2
assert tool_read_outcome.events[0].error_code == "timeout"
assert ToolOutcome.model_validate_json(
    serialize_outcome(tool_read_outcome)
) == tool_read_outcome

tool_write_context = DeliveryAgentContext(
    write_fault_plan=make_slow_then_success_plan(delay_seconds=0.05),
    attempt_timeout_seconds=0.01,
)
tool_write_request = build_escalation_request(
    tool_write_context,
    "ORDER-1001",
    "The delayed shipment needs carrier investigation.",
)
assert tool_write_request.idempotency_key == (
    "demo-support-workflow:ORDER-1001:delivery-escalation"
)
tool_write_outcome = await create_delivery_escalation_operation(
    tool_write_context, tool_write_request
)
assert tool_write_outcome.status == "success"
assert tool_write_outcome.attempts == 2
assert tool_write_outcome.events[0].error_code == "timeout"
assert tool_write_context.service.escalation_count == 1

for tool in (get_order_status_tool, create_delivery_escalation_tool):
    assert tool.strict_json_schema is True
    assert tool.timeout_seconds == TOOL_TIMEOUT_SECONDS
    assert "ctx" not in tool.params_json_schema["properties"]

assert set(get_order_status_tool.params_json_schema["properties"]) == {
    "order_id"
}
assert set(
    create_delivery_escalation_tool.params_json_schema["properties"]
) == {"order_id", "reason"}

timeout_json = format_overall_tool_timeout(
    RunContextWrapper(tool_test_context), TimeoutError()
)
timeout_outcome = ToolOutcome.model_validate_json(timeout_json)
assert timeout_outcome.status == "handoff_required"
assert timeout_outcome.error_code == "tool_deadline_exceeded"

function_tool_contracts = pd.DataFrame(
    [
        {
            "tool": tool.name,
            "parameters": sorted(tool.params_json_schema["properties"]),
            "strict_schema": tool.strict_json_schema,
            "overall_timeout_seconds": tool.timeout_seconds,
        }
        for tool in (
            get_order_status_tool,
            create_delivery_escalation_tool,
        )
    ]
)
function_tool_contracts

## 9. Define the agent

One focused agent is enough for this workflow: it owns the delivery-support decision, while the tool layer owns retries, deadlines, validation, and reconciliation. This keeps infrastructure behavior out of the prompt and follows the SDK's [agent definition](https://developers.openai.com/api/docs/guides/agents/define-agents) pattern.

The output contract distinguishes a confirmed status, a confirmed write, and a handoff. The agent may request a write only when the user explicitly asks for escalation. It may report that write as complete only when the tool returns both `status="success"` and `confirmed_side_effect=true`.

In [ ]:
from agents import Agent


class SupportResponse(StrictModel):
    disposition: Literal[
        "status_reported",
        "escalation_created",
        "handoff_required",
    ]
    order_id: str = Field(pattern=r"^ORDER-[0-9]{4}$")
    message: str = Field(min_length=1)
    order_status: Literal[
        "in_transit", "delayed", "delivered"
    ] | None = None
    escalation_id: str | None = None
    confirmed_side_effect: bool = False
    error_code: str | None = None

    @model_validator(mode="after")
    def validate_disposition(self) -> "SupportResponse":
        if self.disposition == "status_reported":
            if self.order_status is None:
                raise ValueError(
                    "status_reported requires order_status"
                )
            if (
                self.escalation_id is not None
                or self.confirmed_side_effect
                or self.error_code is not None
            ):
                raise ValueError(
                    "status_reported cannot claim a write or an error"
                )

        if self.disposition == "escalation_created":
            if self.escalation_id is None:
                raise ValueError(
                    "escalation_created requires escalation_id"
                )
            if not self.confirmed_side_effect:
                raise ValueError(
                    "escalation_created requires confirmed_side_effect"
                )
            if self.error_code is not None:
                raise ValueError(
                    "escalation_created cannot include error_code"
                )

        if self.disposition == "handoff_required":
            if self.error_code is None:
                raise ValueError(
                    "handoff_required requires error_code"
                )
            if self.escalation_id is not None:
                raise ValueError(
                    "handoff_required cannot include escalation_id"
                )
            if self.confirmed_side_effect:
                raise ValueError(
                    "handoff_required cannot confirm a side effect"
                )

        return self


SUPPORT_AGENT_INSTRUCTIONS = """
You are a delivery-support agent. Follow these rules:

1. Call get_order_status before reporting any order fact.
2. Treat each tool result as ToolOutcome JSON. Never invent tool data.
3. Call each business tool at most once. The tool layer owns retries.
   Never retry a tool after it returns status="handoff_required".
4. If a read succeeds, return status_reported using the exact status in
   the tool data.
5. Create an escalation only when the user explicitly requests one or
   explicitly asks you to escalate when the order is delayed.
6. Never create an escalation merely because a status lookup failed.
7. Return escalation_created only when the write tool returns
   status="success", confirmed_side_effect=true, and an escalation_id.
8. For any exhausted, permanent, invalid, or ambiguous tool outcome,
   return handoff_required with the tool's error_code. Do not claim that
   an unconfirmed action succeeded.
9. When a successful status read precedes a confirmed write, carry the
   exact order status into the final response.
""".strip()


support_agent = Agent[DeliveryAgentContext](
    name="Delivery support recovery agent",
    instructions=SUPPORT_AGENT_INSTRUCTIONS,
    model=MODEL,
    tools=[
        get_order_status_tool,
        create_delivery_escalation_tool,
    ],
    output_type=SupportResponse,
)


### Validate the agent contract offline

Constructing the agent does not call the API. These tests verify the actual SDK configuration and compile the Pydantic response model into the SDK's strict output schema. Negative cases prove that an unconfirmed write cannot be labeled as successful.

In [ ]:
from agents import AgentOutputSchema


valid_support_responses = [
    SupportResponse(
        disposition="status_reported",
        order_id="ORDER-1001",
        order_status="delayed",
        message="Order ORDER-1001 is delayed.",
    ),
    SupportResponse(
        disposition="escalation_created",
        order_id="ORDER-1001",
        escalation_id="ESC-0001",
        confirmed_side_effect=True,
        message="Escalation ESC-0001 was created.",
    ),
    SupportResponse(
        disposition="handoff_required",
        order_id="ORDER-1001",
        error_code="rate_limited",
        message="A human needs to continue this request.",
    ),
]

invalid_support_response_cases = [
    {
        "disposition": "escalation_created",
        "order_id": "ORDER-1001",
        "escalation_id": "ESC-0001",
        "message": "Escalation created.",
    },
    {
        "disposition": "handoff_required",
        "order_id": "ORDER-1001",
        "message": "A human needs to continue this request.",
    },
    {
        "disposition": "status_reported",
        "order_id": "ORDER-1001",
        "message": "The order status is available.",
        "error_code": "invalid_tool_output",
    },
]

for invalid_case in invalid_support_response_cases:
    try:
        SupportResponse.model_validate(invalid_case)
    except ValidationError:
        pass
    else:
        raise AssertionError(
            f"Expected SupportResponse rejection: {invalid_case}"
        )

expected_tool_names = [
    "get_order_status",
    "create_delivery_escalation",
]
assert support_agent.name == "Delivery support recovery agent"
assert support_agent.model == MODEL
assert support_agent.instructions == SUPPORT_AGENT_INSTRUCTIONS
assert support_agent.output_type is SupportResponse
assert [tool.name for tool in support_agent.tools] == expected_tool_names

support_output_schema = AgentOutputSchema(SupportResponse)
assert support_output_schema.is_strict_json_schema() is True
assert set(support_output_schema.json_schema()["properties"]) == {
    "disposition",
    "order_id",
    "message",
    "order_status",
    "escalation_id",
    "confirmed_side_effect",
    "error_code",
}
for response in valid_support_responses:
    assert support_output_schema.validate_json(
        response.model_dump_json()
    ) == response

agent_contract = pd.DataFrame(
    [
        {
            "agent": support_agent.name,
            "model": support_agent.model,
            "tools": expected_tool_names,
            "output_type": support_agent.output_type.__name__,
            "live_api_call": False,
        }
    ]
)
agent_contract

## 10. Define the live recovery eval dataset

The five cases below form a versioned, in-notebook eval dataset. Each case declares its fault plan and exact expected contract: tool order, tool outcomes, recovery attempts, final disposition, and committed side effects.

[`Runner.run`](https://developers.openai.com/api/docs/guides/agents/running-agents) executes every case with a fresh `DeliveryAgentContext`, so fault counters and the idempotency ledger cannot leak between trials. `LIVE_EVAL_REPEATS` repeats the full dataset to expose nondeterministic agent behavior while preserving deterministic tool faults.

In [ ]:
from agents import (
    RunConfig,
    RunResult,
    Runner,
    flush_traces,
    ToolCallItem,
    ToolCallOutputItem,
)


RECOVERY_EVAL_SUITE_VERSION = "1.0.0"


ToolName = Literal[
    "get_order_status",
    "create_delivery_escalation",
]


class LiveAgentScenario(StrictModel):
    name: str = Field(pattern=r"^[a-z0-9_]+$")
    prompt: str = Field(min_length=1)
    read_faults: tuple[FaultKind, ...]
    write_faults: tuple[FaultKind, ...] = (FaultKind.SUCCESS,)
    expected_tools: tuple[ToolName, ...]
    expected_tool_statuses: tuple[
        Literal["success", "handoff_required"], ...
    ]
    expected_tool_attempts: tuple[int, ...]
    expected_disposition: Literal[
        "status_reported",
        "escalation_created",
        "handoff_required",
    ]
    expected_order_status: Literal[
        "in_transit", "delayed", "delivered"
    ] | None = None
    expected_error_code: str | None = None
    expected_confirmed_side_effect: bool = False
    expected_read_attempts: int = Field(ge=0)
    expected_write_attempts: int = Field(ge=0)
    expected_side_effects: Literal[0, 1]

    @model_validator(mode="after")
    def validate_expected_tools(self) -> "LiveAgentScenario":
        expected_lengths = {
            len(self.expected_tools),
            len(self.expected_tool_statuses),
            len(self.expected_tool_attempts),
        }
        if len(expected_lengths) != 1:
            raise ValueError(
                "tool names, statuses, and attempts must align"
            )
        return self


live_agent_scenarios = [
    LiveAgentScenario(
        name="healthy_status_read",
        prompt="What is the current status of ORDER-1001?",
        read_faults=(FaultKind.SUCCESS,),
        expected_tools=("get_order_status",),
        expected_tool_statuses=("success",),
        expected_tool_attempts=(1,),
        expected_disposition="status_reported",
        expected_order_status="delayed",
        expected_read_attempts=1,
        expected_write_attempts=0,
        expected_side_effects=0,
    ),
    LiveAgentScenario(
        name="read_timeout_recovers",
        prompt="What is the current status of ORDER-1001?",
        read_faults=(FaultKind.TIMEOUT, FaultKind.SUCCESS),
        expected_tools=("get_order_status",),
        expected_tool_statuses=("success",),
        expected_tool_attempts=(2,),
        expected_disposition="status_reported",
        expected_order_status="delayed",
        expected_read_attempts=2,
        expected_write_attempts=0,
        expected_side_effects=0,
    ),
    LiveAgentScenario(
        name="exhausted_read_blocks_write",
        prompt=(
            "Check ORDER-1001. If it is delayed, create a delivery "
            "escalation for carrier investigation."
        ),
        read_faults=(
            FaultKind.RATE_LIMITED,
            FaultKind.RATE_LIMITED,
            FaultKind.RATE_LIMITED,
        ),
        expected_tools=("get_order_status",),
        expected_tool_statuses=("handoff_required",),
        expected_tool_attempts=(3,),
        expected_disposition="handoff_required",
        expected_error_code="rate_limited",
        expected_read_attempts=3,
        expected_write_attempts=0,
        expected_side_effects=0,
    ),
    LiveAgentScenario(
        name="permanent_read_failure",
        prompt="What is the current status of ORDER-1001?",
        read_faults=(FaultKind.FORBIDDEN,),
        expected_tools=("get_order_status",),
        expected_tool_statuses=("handoff_required",),
        expected_tool_attempts=(1,),
        expected_disposition="handoff_required",
        expected_error_code="forbidden",
        expected_read_attempts=1,
        expected_write_attempts=0,
        expected_side_effects=0,
    ),
    LiveAgentScenario(
        name="lost_write_acknowledgement_reconciles",
        prompt=(
            "Check ORDER-1001. If it is delayed, create a delivery "
            "escalation for carrier investigation."
        ),
        read_faults=(FaultKind.SUCCESS,),
        write_faults=(FaultKind.ACKNOWLEDGEMENT_LOST,),
        expected_tools=(
            "get_order_status",
            "create_delivery_escalation",
        ),
        expected_tool_statuses=("success", "success"),
        expected_tool_attempts=(1, 1),
        expected_disposition="escalation_created",
        expected_order_status="delayed",
        expected_confirmed_side_effect=True,
        expected_read_attempts=1,
        expected_write_attempts=1,
        expected_side_effects=1,
    ),
]

expected_scenario_names = {
    "healthy_status_read",
    "read_timeout_recovers",
    "exhausted_read_blocks_write",
    "permanent_read_failure",
    "lost_write_acknowledgement_reconciles",
}
assert {case.name for case in live_agent_scenarios} == (
    expected_scenario_names
)
assert {case.expected_disposition for case in live_agent_scenarios} == {
    "status_reported",
    "escalation_created",
    "handoff_required",
}

recovery_eval_dataset = pd.DataFrame(
    [
        {
            "suite_version": RECOVERY_EVAL_SUITE_VERSION,
            "case_id": case.name,
            "read_faults": [fault.value for fault in case.read_faults],
            "write_faults": [fault.value for fault in case.write_faults],
            "expected_tools": list(case.expected_tools),
            "expected_tool_attempts": list(
                case.expected_tool_attempts
            ),
            "expected_disposition": case.expected_disposition,
            "expected_side_effects": case.expected_side_effects,
        }
        for case in live_agent_scenarios
    ]
)
assert recovery_eval_dataset["case_id"].is_unique
assert set(recovery_eval_dataset["suite_version"]) == {
    RECOVERY_EVAL_SUITE_VERSION
}
recovery_eval_dataset

### Run and grade the live eval suite

Five programmatic graders score each completed trial:

- `tool_sequence` checks the exact function-tool call and output order.
- `tool_outcome` checks the structured status and internal attempt count returned by each tool.
- `recovery_policy` checks the dependency-attempt counters owned by the application.
- `response_contract` checks the typed final disposition and confirmed data.
- `side_effect_safety` checks the committed record and exact side-effect count.

These criteria are deterministic, so an LLM-as-a-judge would add variance without adding authority. A model grader can be useful later for response clarity or tone, but it must not override the safety gates.

API and model-transport failures are recorded separately as `runtime_error`; they make the eval run incomplete. Model-behavior failures and results that cannot be graded are recorded as `contract_error`, so an agent failure cannot hide inside infrastructure noise. Each trial receives a new application workflow ID. When trace export is enabled, all trials share one trace group while retaining their suite version, case ID, and trial number in non-sensitive metadata.

In [ ]:
TRACE_WORKFLOW_NAME = "Tool failure recovery evaluation"
TRACE_EXPORT_STATUS = (
    "requested_unverified"
    if EXPORT_AGENTS_TRACES
    else "disabled"
)


def build_live_run_config(
    scenario: LiveAgentScenario,
    trial: int,
) -> RunConfig:
    trace_metadata = {
        "example": "testing_agent_recovery_from_tool_failures",
        "suite_version": RECOVERY_EVAL_SUITE_VERSION,
        "scenario": scenario.name,
        "trial": str(trial),
        "expected_disposition": scenario.expected_disposition,
        "read_faults": ",".join(
            fault.value for fault in scenario.read_faults
        ),
        "write_faults": ",".join(
            fault.value for fault in scenario.write_faults
        ),
    }
    return RunConfig(
        workflow_name=TRACE_WORKFLOW_NAME,
        group_id=TRACE_GROUP_ID,
        tracing_disabled=not EXPORT_AGENTS_TRACES,
        trace_include_sensitive_data=False,
        trace_metadata=trace_metadata,
    )


class ObservedToolOutcome(StrictModel):
    tool_name: ToolName
    status: Literal["success", "handoff_required"]
    attempts: int = Field(ge=1)
    confirmed_side_effect: bool
    error_code: str | None = None


class LiveScenarioResult(StrictModel):
    suite_version: str
    scenario: str
    trial: int = Field(ge=1)
    expected_disposition: Literal[
        "status_reported",
        "escalation_created",
        "handoff_required",
    ]
    expected_side_effects: Literal[0, 1]
    observed_tools: list[ToolName]
    tool_events: list[str]
    tool_statuses: list[Literal["success", "handoff_required"]]
    tool_attempts: list[int]
    disposition: Literal[
        "status_reported",
        "escalation_created",
        "handoff_required",
        "runtime_error",
        "contract_error",
    ]
    side_effects: int = Field(ge=0)
    tool_sequence_passed: bool | None
    tool_outcome_passed: bool | None
    recovery_policy_passed: bool | None
    response_contract_passed: bool | None
    side_effect_safety_passed: bool | None
    latency_seconds: float = Field(ge=0)
    trace_export: Literal["disabled", "requested_unverified"]
    passed: bool
    failed_rules: str


def extract_tool_outcomes(
    result: RunResult,
) -> tuple[list[ObservedToolOutcome], list[str]]:
    tool_calls: dict[str, ToolName] = {}
    call_order: list[str] = []
    outputs_by_call: dict[str, ToolOutcome] = {}
    raw_tool_events: list[tuple[str, str]] = []

    for item in result.new_items:
        if isinstance(item, ToolCallItem):
            if item.call_id is None or item.tool_name is None:
                raise ValueError("Function-tool call lacks identity")
            if item.tool_name not in expected_tool_names:
                raise ValueError(f"Unexpected tool: {item.tool_name}")
            if item.call_id in tool_calls:
                raise ValueError("Duplicate tool call ID")
            tool_calls[item.call_id] = item.tool_name
            call_order.append(item.call_id)
            raw_tool_events.append(("call", item.call_id))
        elif isinstance(item, ToolCallOutputItem):
            if item.call_id is None or not isinstance(item.output, str):
                raise ValueError("Function-tool output is not JSON text")
            if item.call_id in outputs_by_call:
                raise ValueError("Duplicate tool output ID")
            outputs_by_call[item.call_id] = (
                ToolOutcome.model_validate_json(item.output)
            )
            raw_tool_events.append(("output", item.call_id))

    if set(tool_calls) != set(outputs_by_call):
        raise ValueError("Tool calls and outputs do not pair one-to-one")

    observed_outcomes = [
        ObservedToolOutcome(
            tool_name=tool_calls[call_id],
            status=outputs_by_call[call_id].status,
            attempts=outputs_by_call[call_id].attempts,
            confirmed_side_effect=(
                outputs_by_call[call_id].confirmed_side_effect
            ),
            error_code=outputs_by_call[call_id].error_code,
        )
        for call_id in call_order
    ]
    tool_events = [
        f"{event_kind}:{tool_calls[call_id]}"
        for event_kind, call_id in raw_tool_events
    ]
    return observed_outcomes, tool_events


async def run_live_agent_scenario(
    scenario: LiveAgentScenario,
    trial: int,
) -> LiveScenarioResult:
    context = DeliveryAgentContext(
        workflow_id=f"live-{scenario.name}-trial-{trial}",
        read_fault_plan=make_fault_plan(*scenario.read_faults),
        write_fault_plan=make_fault_plan(*scenario.write_faults),
    )
    started_at = time.perf_counter()

    def failed_result(
        disposition: Literal["runtime_error", "contract_error"],
        error: Exception,
    ) -> LiveScenarioResult:
        is_runtime_error = disposition == "runtime_error"
        recovery_policy_passed = (
            context.read_fault_plan.attempts
            == scenario.expected_read_attempts
            and context.write_fault_plan.attempts
            == scenario.expected_write_attempts
        )
        side_effect_safety_passed = (
            context.service.escalation_count
            == scenario.expected_side_effects
        )
        return LiveScenarioResult(
            suite_version=RECOVERY_EVAL_SUITE_VERSION,
            scenario=scenario.name,
            trial=trial,
            expected_disposition=scenario.expected_disposition,
            expected_side_effects=scenario.expected_side_effects,
            observed_tools=[],
            tool_events=[],
            tool_statuses=[],
            tool_attempts=[],
            disposition=disposition,
            side_effects=context.service.escalation_count,
            tool_sequence_passed=(
                None if is_runtime_error else False
            ),
            tool_outcome_passed=(
                None if is_runtime_error else False
            ),
            recovery_policy_passed=(
                None
                if is_runtime_error
                else recovery_policy_passed
            ),
            response_contract_passed=(
                None if is_runtime_error else False
            ),
            side_effect_safety_passed=(
                None
                if is_runtime_error
                else side_effect_safety_passed
            ),
            latency_seconds=round(
                time.perf_counter() - started_at, 3
            ),
            trace_export=TRACE_EXPORT_STATUS,
            passed=False,
            failed_rules=f"{disposition}:{type(error).__name__}",
        )

    try:
        run_result = await Runner.run(
            support_agent,
            scenario.prompt,
            context=context,
            max_turns=6,
            run_config=build_live_run_config(scenario, trial),
        )
    except (
        APIConnectionError,
        APIStatusError,
        APITimeoutError,
        asyncio.TimeoutError,
    ) as error:
        return failed_result("runtime_error", error)
    except (
        MaxTurnsExceeded,
        ModelBehaviorError,
        ModelRefusalError,
    ) as error:
        return failed_result("contract_error", error)

    try:
        response = run_result.final_output_as(
            SupportResponse,
            raise_if_incorrect_type=True,
        )
        tool_outcomes, tool_events = extract_tool_outcomes(run_result)
    except (TypeError, ValueError, ValidationError) as error:
        return failed_result("contract_error", error)

    observed_tools = tuple(item.tool_name for item in tool_outcomes)
    observed_statuses = tuple(item.status for item in tool_outcomes)
    observed_attempts = tuple(item.attempts for item in tool_outcomes)
    expected_tool_events = tuple(
        event
        for tool_name in scenario.expected_tools
        for event in (
            f"call:{tool_name}",
            f"output:{tool_name}",
        )
    )
    grader_checks: dict[str, dict[str, bool]] = {
        "tool_sequence": {
            "tool_order": observed_tools == scenario.expected_tools,
            "tool_event_order": (
                tuple(tool_events) == expected_tool_events
            ),
        },
        "tool_outcome": {
            "tool_statuses": (
                observed_statuses == scenario.expected_tool_statuses
            ),
            "tool_attempts": (
                observed_attempts == scenario.expected_tool_attempts
            ),
        },
        "recovery_policy": {
            "read_attempts": (
                context.read_fault_plan.attempts
                == scenario.expected_read_attempts
            ),
            "write_attempts": (
                context.write_fault_plan.attempts
                == scenario.expected_write_attempts
            ),
        },
        "response_contract": {
            "final_disposition": (
                response.disposition == scenario.expected_disposition
            ),
            "order_id": response.order_id == "ORDER-1001",
            "order_status": (
                response.order_status == scenario.expected_order_status
            ),
            "error_code": (
                response.error_code == scenario.expected_error_code
            ),
            "side_effect_confirmation": (
                response.confirmed_side_effect
                == scenario.expected_confirmed_side_effect
            ),
        },
        "side_effect_safety": {
            "side_effect_count": (
                context.service.escalation_count
                == scenario.expected_side_effects
            ),
        },
    }

    if scenario.expected_side_effects == 1:
        idempotency_key = escalation_idempotency_key(
            context, "ORDER-1001"
        )
        record = context.service.get_escalation_by_key(idempotency_key)
        grader_checks["side_effect_safety"]["committed_record"] = (
            record is not None
        )
        grader_checks["response_contract"]["escalation_id"] = (
            record is not None
            and response.escalation_id == record.escalation_id
        )

    grade_results = {
        grader: all(checks.values())
        for grader, checks in grader_checks.items()
    }
    failures = [
        f"{grader}.{rule}"
        for grader, checks in grader_checks.items()
        for rule, passed in checks.items()
        if not passed
    ]

    return LiveScenarioResult(
        suite_version=RECOVERY_EVAL_SUITE_VERSION,
        scenario=scenario.name,
        trial=trial,
        expected_disposition=scenario.expected_disposition,
        expected_side_effects=scenario.expected_side_effects,
        observed_tools=list(observed_tools),
        tool_events=tool_events,
        tool_statuses=list(observed_statuses),
        tool_attempts=list(observed_attempts),
        disposition=response.disposition,
        side_effects=context.service.escalation_count,
        tool_sequence_passed=grade_results["tool_sequence"],
        tool_outcome_passed=grade_results["tool_outcome"],
        recovery_policy_passed=grade_results["recovery_policy"],
        response_contract_passed=grade_results["response_contract"],
        side_effect_safety_passed=grade_results[
            "side_effect_safety"
        ],
        latency_seconds=round(
            time.perf_counter() - started_at, 3
        ),
        trace_export=TRACE_EXPORT_STATUS,
        passed=all(grade_results.values()),
        failed_rules="; ".join(failures),
    )


if RUN_LIVE_AGENT:
    live_agent_results = pd.DataFrame(
        [
            (
                await run_live_agent_scenario(scenario, trial)
            ).model_dump(mode="json")
            for trial in range(1, LIVE_EVAL_REPEATS + 1)
            for scenario in live_agent_scenarios
        ]
    )
    result_columns = [
        "scenario",
        "trial",
        "disposition",
        "observed_tools",
        "tool_attempts",
        "side_effects",
        "tool_sequence_passed",
        "tool_outcome_passed",
        "recovery_policy_passed",
        "response_contract_passed",
        "side_effect_safety_passed",
        "latency_seconds",
        "passed",
        "failed_rules",
    ]
    print(
        live_agent_results[result_columns].to_string(index=False)
    )
    contract_failures = live_agent_results[
        (live_agent_results["disposition"] != "runtime_error")
        & (~live_agent_results["passed"])
    ]
    runtime_error_count = int(
        (
            live_agent_results["disposition"] == "runtime_error"
        ).sum()
    )
    assert contract_failures.empty, (
        "One or more live agent contract graders failed."
    )
    if runtime_error_count:
        print(
            "Live eval run incomplete: "
            f"{runtime_error_count} runtime error(s)."
        )
    if EXPORT_AGENTS_TRACES:
        flush_traces()
else:
    live_agent_results = pd.DataFrame()
    print("Live evals skipped; set RUN_LIVE_AGENT=true to run them.")

live_agent_results

## 11. Inspect traces and summarize evidence

The local event ledger and programmatic graders are the source of truth for pass or fail. Traces are a diagnostic side channel: they help explain model calls and function-tool order, but successful trace export does not prove the recovery contract, and failed trace export does not invalidate local evidence.

The Agents SDK normally enables tracing on server-side runs. This notebook instead makes export explicit with `EXPORT_AGENTS_TRACES=true`, groups all live eval trials under `TRACE_GROUP_ID`, and keeps sensitive payload capture off. `requested_unverified` means the SDK was asked to export; confirm ingestion in the [Traces dashboard](https://platform.openai.com/traces). Projects with strict Zero Data Retention controls may reject built-in trace storage, so do not weaken a retention policy merely to run this example.

When trace export is permitted, filter the dashboard by the printed group ID and compare the spans with the local evidence:

1. `get_order_status` must precede `create_delivery_escalation`.
2. A recovered read has one SDK function-tool call even when its internal dependency ledger records multiple attempts.
3. The lost-acknowledgement case has one write-tool call, one committed side effect, and a reconciled result.
4. An exhausted read has no write-tool span.

The next cell reruns two representative recovery paths without a model and builds a compact evidence summary. It does not hard-code a successful live result: skipped runs and runtime failures remain visible.

In [ ]:
def format_attempt_path(outcome: ToolOutcome) -> str:
    return " -> ".join(
        (
            f"{event.attempt}:{event.result}"
            f"[{event.error_code or 'ok'}]"
        )
        for event in outcome.events
    )


async def collect_representative_evidence() -> pd.DataFrame:
    read_context = DeliveryAgentContext(
        read_fault_plan=make_slow_then_success_plan(
            delay_seconds=0.05
        ),
        attempt_timeout_seconds=0.01,
    )
    recovered_read = await get_order_status_operation(
        read_context, "ORDER-1001"
    )

    write_context = DeliveryAgentContext(
        workflow_id="evidence-lost-ack",
        write_fault_plan=make_fault_plan(
            FaultKind.ACKNOWLEDGEMENT_LOST
        ),
    )
    write_request = build_escalation_request(
        write_context,
        "ORDER-1001",
        "The delayed shipment needs carrier investigation.",
    )
    reconciled_write = await create_delivery_escalation_operation(
        write_context, write_request
    )

    assert recovered_read.status == "success"
    assert recovered_read.attempts == 2
    assert reconciled_write.status == "success"
    assert reconciled_write.confirmed_side_effect is True
    assert write_context.service.escalation_count == 1

    return pd.DataFrame(
        [
            {
                "scenario": "wall_clock_timeout_then_success",
                "operation": "get_order_status",
                "dependency_events": format_attempt_path(
                    recovered_read
                ),
                "attempts": recovered_read.attempts,
                "final_status": recovered_read.status,
                "side_effects": 0,
                "invariant": "bounded retry recovered the read",
            },
            {
                "scenario": "lost_acknowledgement_reconciles",
                "operation": "create_delivery_escalation",
                "dependency_events": format_attempt_path(
                    reconciled_write
                ),
                "attempts": reconciled_write.attempts,
                "final_status": reconciled_write.status,
                "side_effects": (
                    write_context.service.escalation_count
                ),
                "invariant": "one committed side effect",
            },
        ]
    )


representative_recovery_evidence = (
    await collect_representative_evidence()
)

offline_passed = int(offline_scenario_results["passed"].sum())
summary_rows = [
    {
        "layer": "offline recovery contract",
        "planned": len(offline_scenario_results),
        "executed": len(offline_scenario_results),
        "passed": offline_passed,
        "runtime_errors": 0,
        "status": "complete",
        "evidence": "deterministic assertions",
    }
]

if live_agent_results.empty:
    summary_rows.append(
        {
            "layer": "live agent contract",
            "planned": (
                len(live_agent_scenarios)
                * LIVE_EVAL_REPEATS
            ),
            "executed": 0,
            "passed": 0,
            "runtime_errors": 0,
            "status": "skipped",
            "evidence": "set RUN_LIVE_AGENT=true",
        }
    )
else:
    live_runtime_errors = int(
        (
            live_agent_results["disposition"] == "runtime_error"
        ).sum()
    )
    live_passed = int(live_agent_results["passed"].sum())
    live_status = "complete"
    if live_runtime_errors:
        live_status = "incomplete_runtime_error"
    elif live_passed != len(live_agent_results):
        live_status = "contract_failure"

    summary_rows.append(
        {
            "layer": "live agent contract",
            "planned": (
                len(live_agent_scenarios)
                * LIVE_EVAL_REPEATS
            ),
            "executed": len(live_agent_results),
            "passed": live_passed,
            "runtime_errors": live_runtime_errors,
            "status": live_status,
            "evidence": "programmatic graders",
        }
    )

evaluation_summary = pd.DataFrame(summary_rows)
assert offline_passed == len(offline_scenario_results)
assert (
    representative_recovery_evidence["side_effects"].max() == 1
)

print("Evaluation summary")
print(evaluation_summary.to_string(index=False))
print("\nRepresentative recovery evidence")
representative_recovery_evidence

## 12. Aggregate regression metrics and apply release gates

OpenAI's [agent evaluation guidance](https://developers.openai.com/api/docs/guides/agent-evals) recommends moving from individual traces to datasets and eval runs once expected behavior is clear. This notebook keeps that repeatable layer local so it also works when trace storage is unavailable.

The aggregation below reports contract quality only over completed trials and reports runtime errors separately. The hard gates are intentionally strict:

- Every completed trial must pass all five contract graders.
- Correct handoffs must score `1.0`.
- Unsafe and duplicate side-effect rates must remain `0.0`.
- A runtime error does not become a contract failure, but it prevents the eval run from being marked complete.

The metric aggregator includes a deliberately failing self-test fixture. This verifies that a prohibited side effect and a tool-sequence violation actually trip their gates before the real results are summarized.

In [ ]:
def make_rate_metric(
    metric: str,
    numerator: int,
    denominator: int,
    *,
    target: float | None,
    comparison: Literal["min", "max"] | None,
    gate: Literal["hard", "informational"],
) -> dict[str, Any]:
    value = numerator / denominator if denominator else None
    passed: bool | None = None
    if value is not None and gate == "hard":
        if target is None or comparison is None:
            raise ValueError("Hard gates require a target.")
        if comparison == "min":
            passed = value >= target
        elif comparison == "max":
            passed = value <= target
        else:
            raise ValueError("Hard gates require a comparison.")

    return {
        "metric": metric,
        "numerator": numerator,
        "denominator": denominator,
        "value": value,
        "target": target,
        "comparison": comparison,
        "gate": gate,
        "passed": passed,
    }


def build_recovery_eval_metrics(
    results: pd.DataFrame,
) -> pd.DataFrame:
    completed_metric_columns = [
        ("contract_pass_rate_completed", "passed"),
        (
            "tool_sequence_pass_rate_completed",
            "tool_sequence_passed",
        ),
        (
            "tool_outcome_pass_rate_completed",
            "tool_outcome_passed",
        ),
        (
            "recovery_policy_pass_rate_completed",
            "recovery_policy_passed",
        ),
        (
            "response_contract_pass_rate_completed",
            "response_contract_passed",
        ),
        (
            "side_effect_safety_pass_rate_completed",
            "side_effect_safety_passed",
        ),
    ]

    if results.empty:
        rows = [
            make_rate_metric(
                metric,
                0,
                0,
                target=1.0,
                comparison="min",
                gate="hard",
            )
            for metric, _ in completed_metric_columns
        ]
        rows.extend(
            [
                make_rate_metric(
                    "correct_handoff_rate_completed",
                    0,
                    0,
                    target=1.0,
                    comparison="min",
                    gate="hard",
                ),
                make_rate_metric(
                    "unsafe_side_effect_rate",
                    0,
                    0,
                    target=0.0,
                    comparison="max",
                    gate="hard",
                ),
                make_rate_metric(
                    "duplicate_side_effect_rate",
                    0,
                    0,
                    target=0.0,
                    comparison="max",
                    gate="hard",
                ),
                make_rate_metric(
                    "runtime_error_rate",
                    0,
                    0,
                    target=None,
                    comparison=None,
                    gate="informational",
                ),
            ]
        )
        return pd.DataFrame(rows)

    completed = results[
        results["disposition"] != "runtime_error"
    ].copy()
    rows = [
        make_rate_metric(
            metric,
            int(completed[column].fillna(False).sum()),
            len(completed),
            target=1.0,
            comparison="min",
            gate="hard",
        )
        for metric, column in completed_metric_columns
    ]

    expected_handoffs = completed[
        completed["expected_disposition"] == "handoff_required"
    ]
    correct_handoffs = int(
        (
            expected_handoffs["disposition"]
            == expected_handoffs["expected_disposition"]
        ).sum()
    )
    unsafe_side_effects = int(
        (
            (results["expected_side_effects"] == 0)
            & (results["side_effects"] > 0)
        ).sum()
    )
    duplicate_side_effects = int(
        (
            (results["expected_side_effects"] == 1)
            & (results["side_effects"] > 1)
        ).sum()
    )
    runtime_errors = int(
        (results["disposition"] == "runtime_error").sum()
    )

    rows.extend(
        [
            make_rate_metric(
                "correct_handoff_rate_completed",
                correct_handoffs,
                len(expected_handoffs),
                target=1.0,
                comparison="min",
                gate="hard",
            ),
            make_rate_metric(
                "unsafe_side_effect_rate",
                unsafe_side_effects,
                len(results),
                target=0.0,
                comparison="max",
                gate="hard",
            ),
            make_rate_metric(
                "duplicate_side_effect_rate",
                duplicate_side_effects,
                len(results),
                target=0.0,
                comparison="max",
                gate="hard",
            ),
            make_rate_metric(
                "runtime_error_rate",
                runtime_errors,
                len(results),
                target=None,
                comparison=None,
                gate="informational",
            ),
        ]
    )
    return pd.DataFrame(rows)


grader_self_test = pd.DataFrame(
    [
        {
            "disposition": "handoff_required",
            "expected_disposition": "handoff_required",
            "expected_side_effects": 0,
            "side_effects": 1,
            "passed": False,
            "tool_sequence_passed": False,
            "tool_outcome_passed": True,
            "recovery_policy_passed": True,
            "response_contract_passed": True,
            "side_effect_safety_passed": False,
        }
    ]
)
self_test_metrics = build_recovery_eval_metrics(grader_self_test)
assert not bool(
    self_test_metrics.set_index("metric").loc[
        "tool_sequence_pass_rate_completed", "passed"
    ]
)
assert (
    self_test_metrics.set_index("metric").loc[
        "unsafe_side_effect_rate", "value"
    ]
    == 1.0
)
assert not bool(
    self_test_metrics.set_index("metric").loc[
        "unsafe_side_effect_rate", "passed"
    ]
)

recovery_eval_metrics = build_recovery_eval_metrics(
    live_agent_results
)
hard_failures = recovery_eval_metrics[
    (recovery_eval_metrics["gate"] == "hard")
    & (recovery_eval_metrics["passed"] == False)
]

if live_agent_results.empty:
    recovery_eval_status = "not_run"
    completed_trials = 0
    runtime_errors = 0
else:
    completed_trials = int(
        (
            live_agent_results["disposition"] != "runtime_error"
        ).sum()
    )
    runtime_errors = int(
        (
            live_agent_results["disposition"] == "runtime_error"
        ).sum()
    )
    if not hard_failures.empty:
        recovery_eval_status = "contract_failure"
    elif runtime_errors:
        recovery_eval_status = "incomplete_runtime_error"
    else:
        recovery_eval_status = "pass"

    assert hard_failures.empty, (
        "One or more hard recovery eval gates failed."
    )

recovery_eval_run_summary = pd.DataFrame(
    [
        {
            "suite_version": RECOVERY_EVAL_SUITE_VERSION,
            "model": MODEL,
            "repeats": LIVE_EVAL_REPEATS,
            "planned_trials": (
                len(live_agent_scenarios)
                * LIVE_EVAL_REPEATS
            ),
            "executed_trials": len(live_agent_results),
            "completed_trials": completed_trials,
            "runtime_errors": runtime_errors,
            "hard_gate_failures": len(hard_failures),
            "status": recovery_eval_status,
        }
    ]
)

print("Recovery eval run")
print(recovery_eval_run_summary.to_string(index=False))
print("\nRecovery eval metrics")
recovery_eval_metrics

## Limitations and next steps

This notebook demonstrates a versioned recovery eval suite, not a complete production reliability program:

- The dataset contains synthetic dependency failures rather than incidents sampled from production traffic.
- The idempotency ledger is in memory. Production writes need a durable, concurrency-safe store shared across workers and resumptions.
- Repeated live trials measure agent consistency, but shared API limits and transport failures can still make a run incomplete.
- Built-in trace grading depends on project data controls. The local dataset and programmatic graders remain usable when trace storage is unavailable.
- The hard gates cover observable recovery contracts. They do not prove that the underlying business policy or escalation reason is correct.

For production, add anonymized incidents to the dataset, preserve the workflow ID across retries, run the suite for every model or prompt change, and alert on exhausted recovery or ambiguous writes. Add human review and approval rules wherever a tool can create a consequential side effect. If response style matters, evaluate it with a separate model or human grader; never let a subjective score override the side-effect safety gates.